# URAx-LACE — ALQAC 2026 (one-shot Colab runner)

**Legal Agentic Case-outcome Engine.** Run the cells top-to-bottom.
Recommended runtime: **A100 / L4 GPU** (High-RAM).

Outputs (submission + all artifacts) are saved to your Google Drive under
`ALQAC_RESULT/<run_id>/`. The final file to upload is `[submission] URAx.json`.

> The run is **resumable**: if Colab disconnects, just re-run the last cell — cached
> API evidence on Drive is reused, so **no API calls are wasted**.

## 1. Clone the repository

In [ ]:
import os
if not os.path.exists('ALQAC2026-URAx'):
    !git clone https://github.com/KamonHuiz/ALQAC2026-URAx.git
%cd ALQAC2026-URAx
!git pull -q

## 2. Mount Drive + provide the API token

Token resolution order: `ALQAC_TOKEN` env  ->  Colab Secret `ALQAC_TOKEN`  ->  `ALQAC_RESULT/token.txt` on Drive.
**Do not commit your token.** Easiest: paste it into `MY_TOKEN` below (session-only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

MY_TOKEN = ''  # <-- paste your alqac_... token here (or set a Colab Secret named ALQAC_TOKEN)
if MY_TOKEN:
    os.environ['ALQAC_TOKEN'] = MY_TOKEN

os.makedirs('/content/drive/MyDrive/ALQAC_RESULT/input', exist_ok=True)
print('Put the 60-case PRIVATE TEST json into:')
print('   /content/drive/MyDrive/ALQAC_RESULT/input/')
print('(the pipeline auto-detects any *.json there whose items have a case_query field)')

## 3. Install dependencies (a few minutes on first run)

In [ ]:
!pip install -q -U pip
!pip install -q "vllm>=0.6.3"
!pip install -q -r requirements.txt

## 4. Quick sanity check — offline validation on the labelled public set (no API calls)

Measures outcome accuracy + law micro-F1 on the 50 public cases in a few minutes.
Great for iterating on prompts/modules before spending any API budget.

In [ ]:
!python scripts/run_pipeline.py --split public --no-api --group URAx

## 5. The real run — PRIVATE test (uses the API, ~2-3h due to the 5s rate limit)

Fully resumable. Re-run this cell after any disconnect; cached evidence is reused.

In [ ]:
!python scripts/run_pipeline.py --split private --group URAx

## 6. Inspect the submission

In [ ]:
import json, glob
subs = sorted(glob.glob('/content/drive/MyDrive/ALQAC_RESULT/*/submission.json'))
path = subs[-1]
print('Latest submission:', path)
sub = json.load(open(path, encoding='utf-8'))
print(len(sub), 'cases')
print(json.dumps(sub[0], ensure_ascii=False, indent=2))